# Offline Kaggle inference

Internet must be Off. Attach one private repository bundle, pinned wheels, the pinned base model, the selected PEFT adapter/tokenizer, and official competition data. This notebook creates exactly `/kaggle/working/submission.csv`.

In [ ]:
from pathlib import Path

BUNDLE_GLOB = "/kaggle/input/**/*.bundle"
WHEEL_DIR = "/kaggle/input/pmldl-pinned-wheels"
BASE_MODEL_DIR = "/kaggle/input/pmldl-pinned-base-model"
ADAPTER_DIR = "/kaggle/input/pmldl-selected-adapter"
REPOSITORY_DIR = Path("/kaggle/working/PMLDL-llm-classification-finetuning")
MAX_LENGTH = 1024  # Change to 2048 only for the planned inference-length check.
BATCH_SIZE = 4

In [ ]:
import gc
import glob
import json
import math
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
started = time.perf_counter()

bundles = sorted(glob.glob(BUNDLE_GLOB, recursive=True))
if len(bundles) != 1:
    raise RuntimeError(f"Expected exactly one private repository bundle, found {bundles}")
if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
subprocess.run(["git", "clone", bundles[0], str(REPOSITORY_DIR)], check=True)

if WHEEL_DIR:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--find-links",
            WHEEL_DIR,
            "-r",
            str(REPOSITORY_DIR / "requirements-transformer.lock"),
            "--no-build-isolation",
            "-e",
            str(REPOSITORY_DIR),
        ],
        check=True,
    )
sys.path.insert(0, str(REPOSITORY_DIR / "src"))

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from peft import PeftModel
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig

from pmldl_llm.data import decode_turns, swap_probability_columns
from pmldl_llm.evaluation import normalize_probabilities
from pmldl_llm.truncation import balanced_head_tail_truncate

competition_dirs = [
    path.parent
    for path in Path("/kaggle/input").glob("**/test.csv")
    if (path.parent / "sample_submission.csv").is_file()
]
if len(competition_dirs) != 1:
    raise RuntimeError(f"Expected exactly one competition input, found {competition_dirs}")
data_dir = competition_dirs[0]
test = pd.read_csv(data_dir / "test.csv")
required_test_columns = ["id", "prompt", "response_a", "response_b"]
if not set(required_test_columns).issubset(test.columns):
    raise RuntimeError(f"Unexpected test schema: {test.columns.tolist()}")
if not test["id"].is_unique:
    raise RuntimeError("Test ids must be unique.")

base_model_dir = Path(BASE_MODEL_DIR)
adapter_dir = Path(ADAPTER_DIR)
if not base_model_dir.is_dir() or not adapter_dir.is_dir():
    raise FileNotFoundError(
        f"Attach BASE_MODEL_DIR={base_model_dir} and ADAPTER_DIR={adapter_dir}."
    )

if not torch.cuda.is_available():
    raise RuntimeError("The final inference notebook requires Kaggle GPU.")
compute_dtype = (
    torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
device_count = torch.cuda.device_count()
max_memory = {index: "15GiB" for index in range(device_count)}
tokenizer_source = adapter_dir if (adapter_dir / "tokenizer_config.json").is_file() else base_model_dir
tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_source,
    local_files_only=True,
    use_fast=True,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.sep_token
cls_token_id = tokenizer.cls_token_id
if cls_token_id is None:
    cls_token_id = tokenizer.bos_token_id
sep_token_id = tokenizer.sep_token_id
if sep_token_id is None:
    sep_token_id = tokenizer.eos_token_id
if cls_token_id is None or sep_token_id is None or tokenizer.pad_token_id is None:
    raise RuntimeError("Tokenizer must define CLS/BOS, SEP/EOS and PAD tokens.")

base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_dir,
    local_files_only=True,
    num_labels=3,
    quantization_config=quantization,
    device_map="auto",
    max_memory=max_memory,
    dtype=compute_dtype,
)
model = PeftModel.from_pretrained(
    base_model,
    adapter_dir,
    local_files_only=True,
)
model.config.pad_token_id = int(tokenizer.pad_token_id)
model.config.use_cache = True
model.eval()
input_device = model.get_input_embeddings().weight.device

def render_turns(value):
    rendered = []
    turns = decode_turns(value)
    for index, turn in enumerate(turns):
        if turn is None:
            content = "null_response"
        elif isinstance(turn, str):
            content = turn
        else:
            content = json.dumps(
                turn, ensure_ascii=False, sort_keys=True, separators=(",", ":")
            )
        rendered.append(f"<turn_{index}> {content}")
    return "\n<turn_boundary>\n".join(rendered)

def encode_row(row, swapped):
    response_a = row.response_b if swapped else row.response_a
    response_b = row.response_a if swapped else row.response_b
    prompt_ids = tokenizer.encode(render_turns(row.prompt), add_special_tokens=False)
    response_a_ids = tokenizer.encode(
        render_turns(response_a), add_special_tokens=False
    )
    response_b_ids = tokenizer.encode(
        render_turns(response_b), add_special_tokens=False
    )
    prompt_ids, response_a_ids, response_b_ids, _ = balanced_head_tail_truncate(
        prompt_ids,
        response_a_ids,
        response_b_ids,
        max_length=MAX_LENGTH,
        special_tokens=4,
        budget_weights=(1, 2, 2),
    )
    values = [
        cls_token_id,
        *prompt_ids,
        sep_token_id,
        *response_a_ids,
        sep_token_id,
        *response_b_ids,
        sep_token_id,
    ]
    if len(values) > MAX_LENGTH:
        raise RuntimeError("Encoded sequence exceeds MAX_LENGTH.")
    return values

class TestDataset(Dataset):
    def __init__(self, frame, swapped):
        self.rows = list(frame.itertuples(index=False))
        self.swapped = swapped

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return encode_row(self.rows[index], self.swapped)

def collate(batch):
    longest = min(MAX_LENGTH, int(math.ceil(max(map(len, batch)) / 8.0) * 8))
    input_ids = torch.full(
        (len(batch), longest),
        int(tokenizer.pad_token_id),
        dtype=torch.long,
    )
    attention_mask = torch.zeros((len(batch), longest), dtype=torch.long)
    for index, values in enumerate(batch):
        input_ids[index, : len(values)] = torch.tensor(values, dtype=torch.long)
        attention_mask[index, : len(values)] = 1
    return {"input_ids": input_ids, "attention_mask": attention_mask}

def predict(swapped):
    loader = DataLoader(
        TestDataset(test, swapped),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        collate_fn=collate,
        pin_memory=True,
    )
    chunks = []
    with torch.inference_mode():
        for batch in loader:
            batch = {
                key: value.to(input_device, non_blocking=True)
                for key, value in batch.items()
            }
            with torch.autocast(
                device_type="cuda",
                dtype=compute_dtype,
                enabled=True,
            ):
                logits = model(**batch).logits
            chunks.append(F.softmax(logits.float(), dim=-1).cpu().numpy())
    if not chunks:
        raise RuntimeError("Hidden test is empty.")
    return normalize_probabilities(np.concatenate(chunks, axis=0))

original_probability = predict(swapped=False)
swapped_probability = predict(swapped=True)
swapped_back_probability = swap_probability_columns(swapped_probability)
probability = normalize_probabilities(
    0.5 * (original_probability + swapped_back_probability)
)

submission = pd.DataFrame(
    {
        "id": test["id"].to_numpy(),
        "winner_model_a": probability[:, 0],
        "winner_model_b": probability[:, 1],
        "winner_tie": probability[:, 2],
    }
)
expected_columns = ["id", "winner_model_a", "winner_model_b", "winner_tie"]
if submission.columns.tolist() != expected_columns:
    raise RuntimeError("Submission columns or order are invalid.")
if len(submission) != len(test) or not submission["id"].equals(test["id"]):
    raise RuntimeError("Submission ids do not exactly match test ids.")
values = submission[expected_columns[1:]].to_numpy(dtype=np.float64)
if not np.isfinite(values).all() or (values < 0).any():
    raise RuntimeError("Submission probabilities must be finite and non-negative.")
if not np.allclose(values.sum(axis=1), 1.0, atol=1e-6):
    raise RuntimeError("Submission probability rows must sum to one.")

submission_path = Path("/kaggle/working/submission.csv")
submission.to_csv(submission_path, index=False)
if list(Path("/kaggle/working").glob("submission*.csv")) != [submission_path]:
    raise RuntimeError("Exactly one submission.csv must be produced.")
elapsed = time.perf_counter() - started
if elapsed > 9 * 60 * 60:
    raise RuntimeError(f"Inference exceeded the 9-hour limit: {elapsed:.1f}s")
print("submission.csv:", submission_path)
print("rows:", len(submission))
print("runtime_seconds:", elapsed)
print("actual_compute_dtype:", compute_dtype)
print("gpu_names:", [torch.cuda.get_device_name(i) for i in range(device_count)])